# Step 1.2a — Enriched Feature Matrix
**Thesis: Geopolitical Turning Points and Macroeconomic Volatility  {Extension of Saadaoui (2026)}**

Builds the high-dimensional control matrix for the DoubleML and causal forest steps (Steps 2–5).

---

## Project structure
```
project/
├── data/
│   ├── Saadaoui_2026_JCE.dta
│   └── 02_features/
│       └── raw/
│           ├── bdi.csv         (Investing.com: ; delimited, mm/dd/yyyy, comma decimal)
│           └── gscpi.csv       (NY Fed: ; delimited, French dates, comma decimal)
└── notebooks/
    └── 02_feature_matrix.ipynb
```

## Data sources

| Variable | Source | Format |
|---|---|---|
| Core dataset (PRI, WTI, controls) | Saadaoui (2026) `.dta` | 48 pre-built variables |
| VIX | Yahoo Finance `^VIX` | Daily → monthly mean |
| Gold (USD/oz) | Datahub.io LBMA | Monthly, auto-download |
| Brent crude | FRED `DCOILBRENTEU` | Daily → monthly mean *(used only for Brent-WTI spread; deleted afterwards)* |
| T-bill 3m, 10y Treasury, TED, REER BIS, INDPRO | FRED direct CSV | Monthly, auto-download |
| CNY/USD exchange rate | FRED `DEXCHUS` | Daily → monthly *(level deleted; only Δlog and rolling vol kept)* |
| US credit spread (BAA−10Y) | FRED `BAA10Y` | Monthly, auto-download |
| EM sovereign spread | FRED `BAMLEMCBPIOAS` | Monthly — zero coverage in sample; excluded automatically |
| EM FX volatility | FRED `DTWEXEMEGS` | Daily → 3-month rolling vol of log-changes |
| Baltic Dry Index | Investing.com manual CSV | `;` delimited, `mm/dd/yyyy`, comma decimal |
| GSCPI | NY Fed manual CSV | `;` delimited, French dates, comma decimal |
| Global EPU | Baker-Bloom-Davis Excel | Auto-download |

## Variables in the final feature matrix

**Passed through from `.dta` (Saadaoui baseline):**
- `llwip`, `l2lwip` — L.lwip, L2.lwip (pre-lagged by Saadaoui)
- `dllgop` = `llgop.diff()` — Stata `d.llgop`
- `dl2lgop` = `l2lgop.diff()` — Stata `d.l2lgop`
- `dlpri_*` — alliance controls (pre-computed in `.dta`)

**Derived from new macro series (all lagged 1 month in B4):**
- `term_spread` = `gs10 − tb3ms`
- `lreer` = log(BIS REER) *(coverage starts 1994-02 — see sparse spec note)*
- `dvix` = Δ VIX
- `lgold` = log(gold price)
- `lbdi` = log(BDI)
- `lip` = log(INDPRO)
- `lepu` = log(EPU) *(coverage starts 1997-02 — sparse spec only)*
- `dcny_usd` = Δ log(CNY/USD)
- `cny_vol` = 3-month rolling std of Δ log(CNY/USD), winsorised at 99th pct
- `em_fx_vol` = 3-month rolling std of Δ log(EM FX index) *(starts 2006 — sparse spec only)*
- `brent_wti_spread` = log(Brent)_{t−1} − log(WTI)_{t−1}
- `gscpi` *(coverage starts 1998-02 — sparse spec only)*

## Two control lists for DoubleML

| List | Variables | Effective obs | Use |
|---|---|---|---|
| `controls_all_ml_dense` | baseline + alliance + macro (≥80% coverage only) | ~336 (starts 1994-02) | **Primary DoubleML spec** |
| `controls_all_ml` | dense + sparse (lepu, gscpi, em_fx_vol) | ~191 (starts 2006) | Robustness / post-2006 check |

## Fixes applied in this version
| # | Fix | Details |
|---|-----|---------|
| 1 | Sample end aligned | `2022-01` → `2022-02` (386 obs, matches replication) |
| 2 | `lbrent` leak prevention | Deleted in B3; asserted absent in B4 and B7 |
| 3 | EM FX volatility added | FRED `DTWEXEMEGS` → `em_fx_vol` |
| 4 | `reer` and `indpro` raw levels removed | Deleted from `monthly_series` after deriving `lreer` and `lip`; not in CSV |
| 5 | `cny_vol` winsorised | Capped at 99th percentile — max was 0.234 (CNY revaluation spike) |
| 6 | `controls_all_ml_dense` added | Primary spec excludes sparse vars (coverage <80%) to preserve sample size |
| 7 | Safety assertions extended | B7 now also checks `reer` and `indpro` absent from export |

> **Note on `brent_wti_spread`:** at time *t* the control equals `lbrent_{t−1} − lwti_{t−1}`.
> It is mechanically correlated with lagged `lwti` because oil prices are autocorrelated.
> This is not look-ahead bias but should be noted when interpreting SHAP values.
> The correlation reflects persistence in the WTI series, not a coding error.


---
## Setup

In [1]:
import pandas as pd
import numpy as np
import os
import json
import requests
import warnings
warnings.filterwarnings('ignore')

import yfinance as yf
import pyreadstat
from pathlib import Path

# ── Robust path resolution ─────────────────────────────────────────────────────
# Works whether kernel cwd is project root or notebooks/
cwd = Path.cwd().resolve()
if (cwd / 'notebooks').exists() and (cwd / 'data').exists():
    project_root = cwd
elif cwd.name == 'notebooks' and (cwd.parent / 'data').exists():
    project_root = cwd.parent
else:
    project_root = cwd

RAW  = str((project_root / 'data' / '02_features' / 'raw').resolve())
PROC = str((project_root / 'data' / '02_features').resolve())
DTA  = str((project_root / 'data' / 'Saadaoui_2026_JCE.dta').resolve())

os.makedirs(RAW,  exist_ok=True)
os.makedirs(PROC, exist_ok=True)

# Sample window (buffer before 1990-01 allows lag construction)
START = '1989-01-01'
END   = '2022-03-01'

print('Setup complete.')
print(f'PROJECT_ROOT → {project_root}')
print(f'RAW          → {RAW}')
print(f'PROC         → {PROC}')

Setup complete.
PROJECT_ROOT → C:\Users\HP\Desktop\replication+contribution
RAW          → C:\Users\HP\Desktop\replication+contribution\data\02_features\raw
PROC         → C:\Users\HP\Desktop\replication+contribution\data\02_features


---
## Section A — Load & Download Raw Data
Each cell skips the download if its output file already exists. Safe to re-run from scratch.

### A1. Saadaoui Core Dataset
**Source:** Saadaoui (2026), *Journal of Comparative Economics*  
**File:** `data/Saadaoui_2026_JCE.dta`

**Controls computed here (not in .dta — generated on the fly in Stata with `d.llgop`, `d.l2lgop`):**
- `dllgop`  = `llgop.diff()`  → Stata `d.llgop`
- `dl2lgop` = `l2lgop.diff()` → Stata `d.l2lgop`

In [2]:
if not os.path.exists(DTA):
    raise FileNotFoundError(f'Core dataset not found: {DTA}')

df_core, _ = pyreadstat.read_dta(DTA)

# Convert Stata %tm (months since Jan 1960) to datetime
base = pd.Timestamp('1960-01-01')
df_core['date'] = df_core['Period'].apply(lambda m: base + pd.DateOffset(months=int(m)))
df_core = df_core.set_index('date').sort_index()
df_core.index = df_core.index.to_period('M').to_timestamp()

# ── Compute Stata first-difference controls (not stored in .dta) ───────────────
# Stata: d.llgop  = llgop - L.llgop
# Stata: d.l2lgop = l2lgop - L.l2lgop
df_core['dllgop']  = df_core['llgop'].diff()
df_core['dl2lgop'] = df_core['l2lgop'].diff()

print(f'Core dataset: {df_core.shape[0]} obs, {df_core.shape[1]} variables')
print(f'Range: {df_core.index[0].date()} to {df_core.index[-1].date()}')
print()
print('controls_baseline variables:')
baseline = ['llwip', 'dllgop', 'l2lwip', 'dl2lgop']
print(df_core[baseline].describe().round(4))

Core dataset: 386 obs, 50 variables
Range: 1990-01-01 to 2022-02-01

controls_baseline variables:
          llwip    dllgop    l2lwip   dl2lgop
count  386.0000  385.0000  386.0000  385.0000
mean     4.5735    0.0007    4.5714    0.0007
std      0.2519    0.0120    0.2521    0.0120
min      4.1334   -0.1457    4.1317   -0.1457
25%      4.3750   -0.0041    4.3743   -0.0042
50%      4.6146    0.0012    4.6137    0.0012
75%      4.7972    0.0066    4.7954    0.0066
max      4.9587    0.0438    4.9452    0.0438


### A2. VIX — CBOE Volatility Index
**Source:** Yahoo Finance `^VIX` via `yfinance`  
**Why:** Captures global financial stress and risk-appetite, the key transmission channel from geopolitical shocks to commodity markets (Bloom 2009).  
**Format:** yfinance ≥0.2 saves multi-level headers; loader handles both old and new formats.

In [3]:
P_VIX = f'{RAW}/vix.csv'

if not os.path.exists(P_VIX):
    vix = yf.download('^VIX', start=START, end=END, progress=False)
    if len(vix) > 50:
        if isinstance(vix.columns, pd.MultiIndex):
            vix.columns = [c[0] for c in vix.columns]
        vix[['Close']].rename(columns={'Close': 'vix'}).to_csv(P_VIX)
        print(f'Downloaded VIX: {len(vix)} daily rows')
    else:
        print('VIX download failed')
else:
    print('VIX: file exists, skipping.')

VIX: file exists, skipping.


### A3. Gold Price (USD/oz)
**Source:** Datahub.io — LBMA monthly fixings  
**Why:** Safe-haven asset; gold prices rise during geopolitical uncertainty and capture an alternative transmission channel (Baur & Lucey 2010).  
**Format:** Standard CSV, `Date` + `Price` columns, already monthly.

In [4]:
P_GOLD = f'{RAW}/gold_monthly.csv'

if not os.path.exists(P_GOLD):
    url = 'https://datahub.io/core/gold-prices/_r/-/data/monthly-processed.csv'
    try:
        r = requests.get(url, timeout=20)
        if r.status_code == 200:
            with open(P_GOLD, 'w') as fh:
                fh.write(r.text)
            print('Downloaded Gold: monthly LBMA data')
        else:
            print(f'Gold download failed: HTTP {r.status_code}')
    except Exception as e:
        print(f'Gold download error: {e}')
else:
    print('Gold: file exists, skipping.')

Gold: file exists, skipping.


### A4. Brent Crude Price
**Source:** FRED `DCOILBRENTEU` (Europe Brent Spot Price FOB, USD/barrel)  
**Coverage:** 1987-05 to present — full sample coverage  
**Why Brent-WTI spread and not raw Brent:**  
WTI (`lwti`) is the outcome. Including raw Brent as a control introduces near-perfect collinearity (ρ ≈ 0.99). The **Brent-WTI spread** (log difference) is orthogonal to WTI levels and captures geopolitical supply disruptions that widen the US domestic vs. global price gap (Kilian & Murphy 2014).  
**Format:** FRED direct CSV — clean, no header issues.

In [5]:
P_BRENT = f'{RAW}/brent_fred.csv'

if not os.path.exists(P_BRENT):
    url = 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=DCOILBRENTEU'
    try:
        brent_df = pd.read_csv(url, index_col=0, parse_dates=True, na_values=['.', 'NA', ''])
        brent_df = brent_df[~brent_df.index.duplicated(keep='first')]
        brent_df['DCOILBRENTEU'] = pd.to_numeric(brent_df['DCOILBRENTEU'], errors='coerce')
        brent_df = brent_df.dropna(subset=['DCOILBRENTEU'])
        brent_df[['DCOILBRENTEU']].to_csv(P_BRENT)
        print(f'Downloaded Brent (FRED): {len(brent_df)} daily rows')
        print(f'Coverage: {brent_df.index[0].date()} to {brent_df.index[-1].date()}')
    except Exception as e:
        print(f'FRED Brent download failed: {e}')
else:
    print('Brent (FRED): file exists, skipping.')

Brent (FRED): file exists, skipping.


### A5. FRED Series — Core Macro & Financial Controls
**Source:** FRED direct CSV — no API key required  
**URL pattern:** `https://fred.stlouisfed.org/graph/fredgraph.csv?id=<ID>`

| FRED ID | Variable | Why it belongs |
|---|---|---|
| `TB3MS` | 3-month T-bill yield (%) | US monetary policy stance |
| `GS10` | 10-year Treasury yield (%) | Long-run real rate channel |
| `TEDRATE` | TED spread (%) | Interbank credit risk / financial stress |
| `RBUSBIS` | US REER, BIS broad index | Dollar conditions (oil is dollar-priced) |
| `INDPRO` | US Industrial Production Index | Demand-side oil driver (Kilian 2009) |
| `DEXCHUS` | CNY/USD exchange rate | Direct FX channel of US-China shocks |
| `BAA10Y` | Moody's Baa − 10Y spread (%) | US credit risk premium |
| `BAMLEMCBPIOAS` | EM corporate OAS (bps) | EM financial conditions channel |
| `DTWEXEMEGS` | EM broad FX index | Source for `em_fx_vol` — EM currency stress |


In [6]:
FRED_SERIES = {
    'tb3ms'    : (f'{RAW}/tb3ms.csv',    'TB3MS'),
    'gs10'     : (f'{RAW}/gs10.csv',     'GS10'),
    'tedrate'  : (f'{RAW}/tedrate.csv',  'TEDRATE'),
    'reer_bis' : (f'{RAW}/reer_bis.csv', 'RBUSBIS'),
    'indpro'   : (f'{RAW}/indpro.csv',   'INDPRO'),
    'cny_usd'  : (f'{RAW}/cny_usd.csv',  'DEXCHUS'),
    'baa10y'   : (f'{RAW}/baa10y.csv',   'BAA10Y'),
    'em_oas'   : (f'{RAW}/em_oas.csv',   'BAMLEMCBPIOAS'),
    'em_fx_idx': (f'{RAW}/em_fx_idx.csv','DTWEXEMEGS'),   # EM broad FX index → em_fx_vol
}

for key, (path, fred_id) in FRED_SERIES.items():
    if not os.path.exists(path):
        url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={fred_id}'
        try:
            r = requests.get(url, timeout=20)
            if r.status_code == 200:
                with open(path, 'w') as fh:
                    fh.write(r.text)
                n = len(r.text.strip().split('\n')) - 1
                print(f'Downloaded {key:12s} ({fred_id}): {n} rows')
            else:
                print(f'{key:12s}: HTTP {r.status_code}')
        except Exception as e:
            print(f'{key:12s}: {e}')
    else:
        tmp = pd.read_csv(path, na_values=['.', 'NA', ''])
        print(f'{key:12s}: file exists ({len(tmp)} rows)')


tb3ms       : file exists (1107 rows)
gs10        : file exists (876 rows)
tedrate     : file exists (9407 rows)
reer_bis    : file exists (387 rows)
indpro      : file exists (1287 rows)
cny_usd     : file exists (11816 rows)
baa10y      : file exists (10514 rows)
em_oas      : file exists (794 rows)
em_fx_idx   : file exists (5295 rows)


### A6. Baltic Dry Index (BDI)
**Source:** Investing.com — manual download  
**Place file at:** `data/02_features/raw/bdi.csv`  
**URL:** https://www.investing.com/a88cc4bb-e670-4930-b94c-7f13777f497f  
**Why:** BDI captures global shipping demand — a real-time proxy for world trade and commodity supply-chain conditions (Kilian 2009; Alizadeh & Muradoglu 2014). Geopolitical turning points in US-China trade directly affect shipping volumes.  
**Format:**
- Delimiter: `;`
- Date: `mm/dd/yyyy`
- Values: comma as **decimal** separator (e.g. `2568,3` = 2568.3 — NOT thousands separator)

In [7]:
P_BDI     = f'{RAW}/bdi.csv'
P_BDI_OUT = f'{RAW}/bdi_clean.csv'

if os.path.exists(P_BDI):
    # Read: semicolon delimited, comma is decimal separator
    bdi_raw = pd.read_csv(
        P_BDI,
        sep=';',
        decimal=',',
        header=0,
        na_values=['', ' ', '-']
    )
    # Normalise to exactly 2 columns regardless of what Investing.com puts in the header
    bdi_raw = bdi_raw.iloc[:, :2].copy()
    bdi_raw.columns = ['date', 'bdi']

    # Parse mm/dd/yyyy dates
    bdi_raw['date'] = pd.to_datetime(bdi_raw['date'], format='%m/%d/%Y', errors='coerce')
    bdi_raw['bdi']  = pd.to_numeric(bdi_raw['bdi'], errors='coerce')
    bdi_raw = bdi_raw.dropna(subset=['date', 'bdi']).set_index('date').sort_index()

    bdi_raw[['bdi']].to_csv(P_BDI_OUT)
    print(f'BDI: {len(bdi_raw)} rows | {bdi_raw.index[0].date()} to {bdi_raw.index[-1].date()}')
    print(f'     Value range: {bdi_raw["bdi"].min():.0f} – {bdi_raw["bdi"].max():.0f}')
else:
    print(f'BDI file not found — download manually.')
    print(f'URL: https://www.investing.com/a88cc4bb-e670-4930-b94c-7f13777f497f')
    print(f'Save as: {P_BDI}')

BDI file not found — download manually.
URL: https://www.investing.com/a88cc4bb-e670-4930-b94c-7f13777f497f
Save as: C:\Users\HP\Desktop\replication+contribution\data\02_features\raw/bdi.csv


### A7. Global Supply Chain Pressure Index (GSCPI)
**Source:** Federal Reserve Bank of New York  
**Place file at:** `data/02_features/raw/gscpi.csv`  
**URL:** https://www.newyorkfed.org/research/policy/gscpi  
**Why:** Measures global supply chain disruptions directly. Geopolitical turning points (trade wars, sanctions) cause supply chain stress — this is the key *transmission variable* (Benigno et al. 2022).  
**Coverage:** 1998-01 to present  
**Format:**
- Delimiter: `;`
- Dates: French abbreviations, mixed 2/4-digit years (e.g. `31-janv-98`, `28-Feb-1998`)
- Values: comma as **decimal** separator (e.g. `-1,09` = −1.09)

In [8]:
P_GSCPI     = f'{RAW}/gscpi.csv'
P_GSCPI_OUT = f'{RAW}/gscpi_cleaned.csv'
gscpi_series = None

if os.path.exists(P_GSCPI):
    # Read without skiprows — keep all data rows
    gscpi_df = pd.read_csv(P_GSCPI, sep=';', header=0, usecols=[0, 1],
                            na_values=['', ' ', '-'])
    date_col  = gscpi_df.columns[0]
    value_col = gscpi_df.columns[1]

    # French → English month abbreviations
    FR_TO_EN = {
        'janv': 'Jan', 'fév': 'Feb', 'févr': 'Feb',
        'mars': 'Mar', 'avr': 'Apr', 'mai': 'May',
        'juin': 'Jun', 'juil': 'Jul', 'août': 'Aug', 'aout': 'Aug',
        'sept': 'Sep', 'oct': 'Oct', 'nov': 'Nov',
        'déc': 'Dec', 'dec': 'Dec',
    }

    def parse_gscpi_date(s):
        s = str(s).strip().lower()
        for fr, en in FR_TO_EN.items():
            s = s.replace(fr, en)
        s = s.title()
        dt = pd.to_datetime(s, dayfirst=True, errors='coerce')
        # pandas maps 2-digit years 69–99 → 1969–1999, 00–68 → 2000–2068
        # GSCPI starts 1998, so any year > 2030 means a 2-digit year was mis-mapped
        if pd.notna(dt) and dt.year > 2030:
            dt = dt.replace(year=dt.year - 100)
        return dt

    gscpi_df[date_col]  = gscpi_df[date_col].apply(parse_gscpi_date)
    gscpi_df[value_col] = (gscpi_df[value_col].astype(str)
                            .str.replace(',', '.', regex=False)
                            .pipe(pd.to_numeric, errors='coerce'))

    gscpi_df = (gscpi_df
                .dropna(subset=[date_col, value_col])
                .set_index(date_col)
                .sort_index())

    gscpi_series = gscpi_df[value_col].resample('MS').last().rename('gscpi')
    gscpi_series.to_csv(P_GSCPI_OUT)

    print(f'GSCPI: {gscpi_series.notna().sum()} months '
          f'| {gscpi_series.first_valid_index().date()} → {gscpi_series.last_valid_index().date()}')
    print(f'       Original kept at {P_GSCPI}')
    print(f'       Cleaned saved to {P_GSCPI_OUT}')
else:
    print(f'GSCPI not found — will be excluded from feature matrix.')
    print(f'Download from: https://www.newyorkfed.org/research/policy/gscpi')
    print(f'Save as: {P_GSCPI}')

GSCPI not found — will be excluded from feature matrix.
Download from: https://www.newyorkfed.org/research/policy/gscpi
Save as: C:\Users\HP\Desktop\replication+contribution\data\02_features\raw/gscpi.csv


### A8. Global Economic Policy Uncertainty Index (EPU)
**Source:** Baker, Bloom & Davis — policyuncertainty.com  
**URL:** `https://www.policyuncertainty.com/media/Global_Policy_Uncertainty_Data.xlsx`  
**Why:** Captures uncertainty about economic *policy* globally — distinct from financial volatility (VIX). Directly relevant to Saadaoui's research question: geopolitical turning points generate policy uncertainty that transmits to commodity markets (Baker, Bloom & Davis 2016).  
**Coverage:** Monthly from 1997-01

In [9]:
P_EPU = f'{RAW}/epu_global.xlsx'

if not os.path.exists(P_EPU):
    url = 'https://www.policyuncertainty.com/media/Global_Policy_Uncertainty_Data.xlsx'
    try:
        r = requests.get(url, timeout=30)
        if r.status_code == 200:
            with open(P_EPU, 'wb') as fh:
                fh.write(r.content)
            print('Downloaded EPU global index')
        else:
            print(f'EPU download failed: HTTP {r.status_code}')
    except Exception as e:
        print(f'EPU download error: {e}')
else:
    print('EPU: file exists, skipping.')

EPU: file exists, skipping.


### A9. CDS Spreads — US and China 5-Year Sovereign
**Source:** Manual download from Bloomberg, Refinitiv, or Investing.com  
**Place files at:** `data/02_features/raw/us_cds_5y.csv` and `data/02_features/raw/china_cds_5y.csv`  
**Why:** Sovereign CDS directly price geopolitical risk. US-China tension widening their respective CDS spreads is a direct financial-market signal of geopolitical stress — exactly the transmission channel the thesis is studying.  
**Expected format:** Any delimiter, first column = date, second column = CDS value in basis points.

In [10]:
# Sample window for these CDS files has limited coverage over our window frame
CDS_FILES = {
    'us_cds_5y'   : f'{RAW}/us_cds_5y.csv',
    'china_cds_5y': f'{RAW}/china_cds_5y.csv',
}

cds_series_store = {}

for name, path in CDS_FILES.items():
    if os.path.exists(path):
        try:
            cds_df = pd.read_csv(path)
            cds_df = cds_df.iloc[:, :2].copy()
            cds_df.columns = ['date', name]
            cds_df['date'] = pd.to_datetime(
                cds_df['date'].astype(str).str.replace(',', '.', regex=False),
                dayfirst=True, errors='coerce'
            )
            cds_df[name] = pd.to_numeric(
                cds_df[name].astype(str).str.replace(',', '.', regex=False),
                errors='coerce'
            )
            cds_df = cds_df.dropna(subset=['date']).set_index('date').sort_index()
            cds_m  = cds_df[name].resample('MS').last().rename(name)
            cds_series_store[name] = cds_m
            print(f'{name}: {cds_m.notna().sum()} months | {cds_m.first_valid_index().date()} → {cds_m.last_valid_index().date()}')
        except Exception as e:
            print(f'{name}: parse failed ({e})')
    else:
        print(f'{name}: not found (optional — add file to use)')

us_cds_5y: not found (optional — add file to use)
china_cds_5y: not found (optional — add file to use)


---
## Section B — Merge & Build Feature Matrix

### B1. Helper Functions

In [11]:
def ensure_datetimeindex(s):
    """Ensure a Series or DataFrame has a clean DatetimeIndex."""
    s = s.copy()
    if not isinstance(s.index, pd.DatetimeIndex):
        s.index = pd.to_datetime(s.index, errors='coerce')
    s = s[s.index.notna()]
    s = s[~s.index.duplicated(keep='first')]
    return s.sort_index()

def to_monthly_mean(s):
    """Resample a daily Series to monthly mean."""
    s = ensure_datetimeindex(s)
    s = pd.to_numeric(s, errors='coerce')
    return s.resample('MS').mean()

def to_monthly_last(s):
    """Resample a Series to month-start last value."""
    s = ensure_datetimeindex(s)
    s = pd.to_numeric(s, errors='coerce')
    return s.resample('MS').last()

def load_fred(path, col_name):
    """Load a standard FRED CSV and return a monthly Series."""
    df = pd.read_csv(path, index_col=0, parse_dates=True, na_values=['.', 'NA', ''])
    df = ensure_datetimeindex(df)
    s  = pd.to_numeric(df.iloc[:, 0], errors='coerce')
    return s.resample('MS').last().rename(col_name)

print('Helper functions defined.')

Helper functions defined.


### B2. Build All Monthly Series

In [12]:
monthly_series = {}

# ── VIX ───────────────────────────────────────────────────────────────────────
if os.path.exists(f'{RAW}/vix.csv'):
    vix_d = pd.read_csv(f'{RAW}/vix.csv', index_col=0,
                         skiprows=lambda i: i in [1, 2], na_values=['.', 'NA', ''])
    vix_d = ensure_datetimeindex(vix_d)
    pcol  = [c for c in vix_d.columns if c.lower() in ('close', 'vix', 'adj close')][0]
    monthly_series['vix'] = to_monthly_mean(vix_d[pcol]).rename('vix')
    print(f"vix          : {monthly_series['vix'].notna().sum()} months")

# ── Gold ──────────────────────────────────────────────────────────────────────
if os.path.exists(f'{RAW}/gold_monthly.csv'):
    gold_d = pd.read_csv(f'{RAW}/gold_monthly.csv', index_col=0, parse_dates=True)
    gold_d = ensure_datetimeindex(gold_d)
    gold_s = to_monthly_last(gold_d.iloc[:, 0])
    monthly_series['lgold'] = np.log(gold_s.replace(0, np.nan)).rename('lgold')
    print(f"lgold        : {monthly_series['lgold'].notna().sum()} months")

# ── Brent (FRED DCOILBRENTEU) TEMPORARY: only used to compute brent_wti_spread in B3
# lbrent is deleted from monthly_series immediately after brent_wti_spread is created.
# It must NOT appear in df_enriched. An assertion in B4 enforces this.
if os.path.exists(f'{RAW}/brent_fred.csv'):
    brent_d = pd.read_csv(f'{RAW}/brent_fred.csv', index_col=0,
                           parse_dates=True, na_values=['.', 'NA', ''])
    brent_d = ensure_datetimeindex(brent_d)
    brent_d['DCOILBRENTEU'] = pd.to_numeric(brent_d['DCOILBRENTEU'], errors='coerce')
    brent_m = to_monthly_mean(brent_d['DCOILBRENTEU'])
    monthly_series['lbrent'] = np.log(brent_m.replace(0, np.nan)).rename('lbrent')
    print(f"lbrent       : {monthly_series['lbrent'].notna().sum()} months  [TEMPORARY — deleted after B3]")
else:
    print("lbrent       : brent_fred.csv not found — run A4 first")

# ── FRED core series ──────────────────────────────────────────────────────────
fred_load = {
    'tb3ms'    : f'{RAW}/tb3ms.csv',
    'gs10'     : f'{RAW}/gs10.csv',
    'tedrate'  : f'{RAW}/tedrate.csv',
    'reer'     : f'{RAW}/reer_bis.csv',
    'indpro'   : f'{RAW}/indpro.csv',
    'cny_usd'  : f'{RAW}/cny_usd.csv',
    'us_spread': f'{RAW}/baa10y.csv',
    'em_oas'   : f'{RAW}/em_oas.csv',
    'em_fx_idx': f'{RAW}/em_fx_idx.csv',   
}
for col, path in fred_load.items():
    if os.path.exists(path):
        monthly_series[col] = load_fred(path, col)
        print(f"{col:14s}: {monthly_series[col].notna().sum()} months")

# ── BDI ───────────────────────────────────────────────────────────────────────
if os.path.exists(f'{RAW}/bdi_clean.csv'):
    bdi_d = pd.read_csv(f'{RAW}/bdi_clean.csv', index_col=0, parse_dates=True)
    bdi_d = ensure_datetimeindex(bdi_d)
    monthly_series['bdi'] = to_monthly_mean(bdi_d['bdi']).rename('bdi')
    print(f"bdi          : {monthly_series['bdi'].notna().sum()} months")

# ── GSCPI ─────────────────────────────────────────────────────────────────────
if gscpi_series is not None and gscpi_series.notna().any():
    monthly_series['gscpi'] = gscpi_series
    print(f"gscpi        : {monthly_series['gscpi'].notna().sum()} months")

# ── EPU ───────────────────────────────────────────────────────────────────────
if os.path.exists(f'{RAW}/epu_global.xlsx'):
    try:
        epu_df = pd.read_excel(f'{RAW}/epu_global.xlsx', header=None)
        header_row = epu_df[epu_df[0] == 'Year'].index[0]
        epu_df = pd.read_excel(f'{RAW}/epu_global.xlsx', skiprows=header_row)
        epu_df.columns = ['Year', 'Month', 'GEPU_current', 'GEPU_ppp']
        epu_df['Year']  = pd.to_numeric(epu_df['Year'],  errors='coerce')
        epu_df['Month'] = pd.to_numeric(epu_df['Month'], errors='coerce')
        epu_df = epu_df.dropna(subset=['Year', 'Month'])
        epu_df['date'] = pd.to_datetime(
            epu_df[['Year', 'Month']].assign(Day=1)
            .rename(columns={'Year': 'year', 'Month': 'month', 'Day': 'day'})
        )
        epu_col = 'GEPU_ppp' if 'GEPU_ppp' in epu_df.columns else 'GEPU_current'
        epu_s = epu_df.set_index('date')[epu_col].sort_index()
        epu_s = ensure_datetimeindex(epu_s)
        monthly_series['epu'] = epu_s.resample('MS').last().rename('epu')
        print(f"epu          : {monthly_series['epu'].notna().sum()} months")
    except Exception as e:
        print(f'EPU load error: {e}')

# ── Optional CDS spreads (if files were provided) ─────────────────────────────
for name, cds_s in cds_series_store.items():
    monthly_series[name] = cds_s
    print(f"{name:14s}: {cds_s.notna().sum()} months")

print(f'\nTotal series collected (including lbrent temporary): {len(monthly_series)}')


vix          : 386 months
lgold        : 2319 months
lbrent       : 468 months  [TEMPORARY — deleted after B3]
tb3ms         : 1107 months
gs10          : 876 months
tedrate       : 433 months
reer          : 387 months
indpro        : 1287 months
cny_usd       : 544 months
us_spread     : 484 months
em_oas        : 37 months
em_fx_idx     : 244 months
bdi          : 462 months
epu          : 347 months

Total series collected (including lbrent temporary): 14


### B3. Derived Variables

In [13]:
derived = []

# ── Yield curve slope ─────────────────────────────────────────────────────────
if 'gs10' in monthly_series and 'tb3ms' in monthly_series:
    monthly_series['term_spread'] = (monthly_series['gs10'] - monthly_series['tb3ms']).rename('term_spread')
    derived.append('term_spread')

# ── Log REER, then DELETE raw reer level ──────────────────────────────────────
# Why delete reer: the raw BIS index level is I(1) and redundant once we log it.
# Keeping reer alongside lreer in the CSV would create a collinear column that
# could confuse any user loading the CSV directly without using var_roles.json.
if 'reer' in monthly_series:
    monthly_series['lreer'] = np.log(monthly_series['reer'].replace(0, np.nan)).rename('lreer')
    derived.append('lreer')
    del monthly_series['reer']   # raw level dropped — only lreer enters the CSV
    print('reer deleted from monthly_series — lreer retained.')

# ── VIX first difference (fear shock) ─────────────────────────────────────────
if 'vix' in monthly_series:
    monthly_series['dvix'] = monthly_series['vix'].diff().rename('dvix')
    derived.append('dvix')

# ── Log BDI ───────────────────────────────────────────────────────────────────
if 'bdi' in monthly_series:
    monthly_series['lbdi'] = np.log(monthly_series['bdi'].replace(0, np.nan)).rename('lbdi')
    derived.append('lbdi')
    del monthly_series['bdi']    # raw level dropped — only lbdi enters the CSV
    print('bdi deleted from monthly_series — lbdi retained.')

# ── Log Industrial Production, then DELETE raw indpro ─────────────────────────
# Why delete indpro: same reason as reer — keeping both the raw level and its
# log creates a perfectly collinear pair. lip is what the plan calls for.
if 'indpro' in monthly_series:
    monthly_series['lip'] = np.log(monthly_series['indpro'].replace(0, np.nan)).rename('lip')
    derived.append('lip')
    del monthly_series['indpro']  # raw level dropped — only lip enters the CSV
    print('indpro deleted from monthly_series — lip retained.')

# ── Log EPU, then DELETE raw epu ─────────────────────────────────────────────
if 'epu' in monthly_series:
    monthly_series['lepu'] = np.log(monthly_series['epu'].replace(0, np.nan)).rename('lepu')
    derived.append('lepu')
    del monthly_series['epu']    # raw level dropped — only lepu enters the CSV
    print('epu deleted from monthly_series — lepu retained.')

# ── CNY/USD: Δlog and 3-month rolling vol (winsorised) ───────────────────────
# Why winsorise cny_vol: the raw rolling std reaches 0.234 (≈24% monthly vol)
# due to the July 2005 or August 2015 CNY revaluation events. A single extreme
# month dominates the 3-month window and inflates the series.
# We cap at the 99th percentile of the full time series before merging.
# dcny_usd is NOT winsorised — individual monthly returns are well-behaved.
if 'cny_usd' in monthly_series:
    cny_log = np.log(monthly_series['cny_usd'].replace(0, np.nan))
    monthly_series['dcny_usd'] = cny_log.diff().rename('dcny_usd')
    cny_vol_raw = cny_log.diff().rolling(3, min_periods=2).std()
    p99 = cny_vol_raw.quantile(0.99)
    monthly_series['cny_vol'] = cny_vol_raw.clip(upper=p99).rename('cny_vol')
    derived.extend(['dcny_usd', 'cny_vol'])
    del monthly_series['cny_usd']
    print(f'cny_vol winsorised at 99th pct = {p99:.4f}  (raw max was {cny_vol_raw.max():.4f})')

# ── EM FX volatility (FRED DTWEXEMEGS — broad EM FX index) ───────────────────
# Why include: the plan explicitly asks for "currency volatility indices for CNY
# and other EM currencies". DTWEXEMEGS is the Fed's broad EM currency index;
# its 3-month rolling log-change vol captures aggregate EM FX stress.
# Coverage starts ~2006 so this variable goes into controls_all_ml (robustness)
# but NOT into controls_all_ml_dense (primary spec).
if 'em_fx_idx' in monthly_series:
    em_fx_log = np.log(monthly_series['em_fx_idx'].replace(0, np.nan))
    em_fx_ret  = em_fx_log.diff()
    monthly_series['em_fx_vol'] = em_fx_ret.rolling(3, min_periods=2).std().rename('em_fx_vol')
    derived.append('em_fx_vol')
    del monthly_series['em_fx_idx']
    print(f"em_fx_vol    : {monthly_series['em_fx_vol'].notna().sum()} months")

# ── Brent-WTI spread (log difference) ────────────────────────────────────────
# Why brent_wti_spread instead of raw lbrent:
# Raw log Brent (lbrent) has ρ≈0.99 with the outcome lwti — including it as a
# control would dominate any ML model and absorb the treatment signal.
# The spread (lbrent − lwti) is orthogonal to WTI levels in expectation and
# captures supply disruptions that widen the US domestic vs. global price gap
# (Kilian & Murphy 2014). After the spread is built, lbrent is deleted.
# Note: the spread is lagged 1 month in B4, so at time t the model sees
# lbrent_{t-1} − lwti_{t-1}, which is correlated with lwti_{t-1} due to
# oil price autocorrelation. This is documented and not a look-ahead leak.
if 'lbrent' in monthly_series:
    lwti_s = ensure_datetimeindex(df_core['lwti'].copy())
    lwti_s.index = lwti_s.index.to_period('M').to_timestamp()
    lbrent_aligned = monthly_series['lbrent'].reindex(lwti_s.index)
    monthly_series['brent_wti_spread'] = (lbrent_aligned - lwti_s).rename('brent_wti_spread')
    del monthly_series['lbrent']
    derived.append('brent_wti_spread')
    print('lbrent deleted from monthly_series — brent_wti_spread retained.')

print(f'\nDerived variables added: {derived}')
print(f'monthly_series keys after B3: {sorted(monthly_series.keys())}')

# # Confirm all raw levels are gone
# for must_be_gone in ['lbrent', 'cny_usd', 'em_fx_idx', 'reer', 'indpro', 'epu', 'bdi']:
#     assert must_be_gone not in monthly_series, f'{must_be_gone} still in monthly_series\'
# print('✓ All raw/intermediate series confirmed absent from monthly_series')


reer deleted from monthly_series — lreer retained.
bdi deleted from monthly_series — lbdi retained.
indpro deleted from monthly_series — lip retained.
epu deleted from monthly_series — lepu retained.
cny_vol winsorised at 99th pct = 0.1163  (raw max was 0.2342)
em_fx_vol    : 242 months
lbrent deleted from monthly_series — brent_wti_spread retained.

Derived variables added: ['term_spread', 'lreer', 'dvix', 'lbdi', 'lip', 'lepu', 'dcny_usd', 'cny_vol', 'em_fx_vol', 'brent_wti_spread']
monthly_series keys after B3: ['brent_wti_spread', 'cny_vol', 'dcny_usd', 'dvix', 'em_fx_vol', 'em_oas', 'gs10', 'lbdi', 'lepu', 'lgold', 'lip', 'lreer', 'tb3ms', 'tedrate', 'term_spread', 'us_spread', 'vix']


### B4. Merge with Core Dataset

> **Look-ahead bias prevention:** All newly added macro controls are lagged 1 month (`shift(1)`) before merging. At time *t*, the model sees only controls known at *t−1*. This is critical for causal validity of the DoubleML step (see Plan Step 10).  
> **The Saadaoui baseline controls (`llwip`, `dllgop`, `l2lwip`, `dl2lgop`) are already pre-lagged in the `.dta` file — do NOT lag them again.**

In [14]:
# Align core index to month-start timestamps
df_core.index = pd.to_datetime(df_core.index).to_period('M').to_timestamp()
df_enriched = df_core.copy()

for col_name, series in monthly_series.items():
    s = series.copy()
    s = ensure_datetimeindex(s)
    s.index = s.index.to_period('M').to_timestamp()
    s.name = col_name
    df_enriched = df_enriched.join(s, how='left')

# ── SAFETY CHECKS post-merge ──────────────────────────────────────────────────
for must_be_gone in ['lbrent', 'cny_usd', 'em_fx_idx', 'reer', 'indpro', 'epu', 'bdi']:
    assert must_be_gone not in df_enriched.columns, (
        f"CRITICAL: {must_be_gone} leaked into df_enriched. Re-run from B3."
    )
print('✓ No raw/intermediate columns in df_enriched')

# Lag all new macro controls by 1 month (look-ahead bias prevention).
# The Saadaoui baseline controls (llwip, dllgop, l2lwip, dl2lgop) are
# already pre-lagged in the .dta — do NOT lag them again.
new_ctrl_cols = [c for c in monthly_series.keys() if c in df_enriched.columns]
df_enriched[new_ctrl_cols] = df_enriched[new_ctrl_cols].shift(1)

# Restrict to Saadaoui sample: 1990-01 to 2022-02 (386 obs — matches replication notebook).
df_enriched = df_enriched.loc['1990-01-01':'2022-02-01']

print(f'Enriched dataset: {df_enriched.shape[0]} obs × {df_enriched.shape[1]} variables')
print(f'Range: {df_enriched.index[0].date()} to {df_enriched.index[-1].date()}')
print('(Expected: 386 obs, 1990-01-01 to 2022-02-01)')


✓ No raw/intermediate columns in df_enriched
Enriched dataset: 386 obs × 67 variables
Range: 1990-01-01 to 2022-02-01
(Expected: 386 obs, 1990-01-01 to 2022-02-01)


### B5. Coverage Audit

In [15]:
new_vars = [c for c in monthly_series.keys() if c in df_enriched.columns]

audit = pd.DataFrame({
    'n_obs'     : df_enriched[new_vars].notna().sum(),
    'pct_filled': (df_enriched[new_vars].notna().mean() * 100).round(1),
    'first_obs' : df_enriched[new_vars].apply(lambda c: c.first_valid_index()),
    'last_obs'  : df_enriched[new_vars].apply(lambda c: c.last_valid_index()),
}).sort_values('pct_filled', ascending=False)

print(audit.to_string())

dead_vars = audit[audit['pct_filled'] == 0].index.tolist()
if dead_vars:
    print(f'\n⚠  ZERO-COVERAGE (excluded from all specs): {dead_vars}')

sparse_audit = audit[(audit['pct_filled'] > 0) & (audit['pct_filled'] < 80)].index.tolist()
if sparse_audit:
    print(f'\n⚠  SPARSE (<80% — robustness spec only): {sparse_audit}')
    print('   Including these in the primary DoubleML spec reduces effective obs.')
    for v in sparse_audit:
        row = audit.loc[v]
        print(f'   {v}: {int(row["n_obs"])} obs ({row["pct_filled"]}%) from {row["first_obs"]}')

# Confirm no raw levels survived to this point
for should_be_absent in ['reer', 'indpro', 'epu', 'bdi', 'lbrent', 'cny_usd', 'em_fx_idx']:
    if should_be_absent in df_enriched.columns:
        print(f'\n⚠  WARNING: {should_be_absent} still in df_enriched — should have been deleted in B3')


                  n_obs  pct_filled  first_obs   last_obs
vix                 385        99.7 1990-02-01 2022-02-01
lgold               385        99.7 1990-02-01 2022-02-01
tb3ms               385        99.7 1990-02-01 2022-02-01
gs10                385        99.7 1990-02-01 2022-02-01
tedrate             385        99.7 1990-02-01 2022-02-01
us_spread           385        99.7 1990-02-01 2022-02-01
term_spread         385        99.7 1990-02-01 2022-02-01
lip                 385        99.7 1990-02-01 2022-02-01
lbdi                385        99.7 1990-02-01 2022-02-01
brent_wti_spread    385        99.7 1990-02-01 2022-02-01
dcny_usd            385        99.7 1990-02-01 2022-02-01
cny_vol             385        99.7 1990-02-01 2022-02-01
dvix                384        99.5 1990-03-01 2022-02-01
lreer               337        87.3 1994-02-01 2022-02-01
lepu                301        78.0 1997-02-01 2022-02-01
em_fx_vol           191        49.5 2006-04-01 2022-02-01
em_oas        

### B6. Variable Role Dictionary

> Zero-coverage variables are automatically excluded from `controls_macro`.  
> Sparse variables (coverage <80%) are kept in `controls_macro` but flagged.

In [16]:
# Variables with zero coverage — exclude from all ML specs
zero_coverage = [c for c in new_vars
                 if df_enriched[c].notna().sum() == 0]

# Sparse variables: present but <80% coverage.
# These are NOT used in the primary DoubleML spec because including them
# reduces the effective sample size — the sparsest (em_fx_vol, starts 2006)
# would cut observations from 386 to ~191, halving the sample.
# They are retained in controls_all_ml for post-2006 robustness checks only.
sparse_vars = [
    c for c in new_vars
    if 0 < df_enriched[c].notna().mean() * 100 < 80
    and c not in zero_coverage
]

VAR_ROLES = {
    'outcome'           : ['lwti'],
    'treatment'         : ['lpri'],
    'instrument'        : ['d2pri', 'd2pri_jp'],

    # Exact Saadaoui (2026) controls from Stata do-file:
    # locproj lwti lpri llwip d.llgop l2lwip d.l2lgop
    # dllgop  = llgop.diff()   (not stored in .dta — computed in A1)
    # dl2lgop = l2lgop.diff()  (not stored in .dta — computed in A1)
    'controls_baseline' : ['llwip', 'dllgop', 'l2lwip', 'dl2lgop'],

    # Alliance controls: dlpri_* pre-computed by Saadaoui and stored in .dta.
    # Plain 'dlpri' (US-China first difference) is intentionally excluded —
    # lpri is already the treatment variable; its first difference has no role
    # as a control in the US-China specification.
    'controls_alliance' : [c for c in df_enriched.columns if c.startswith('dlpri_')],

    # Dense macro controls: ≥80% sample coverage.
    # Safe to include in the primary DoubleML spec without losing observations.
    # lreer starts 1994-02 (87.3%) — primary spec effective sample starts ~1994.
    # em_oas excluded (zero coverage). Sparse vars (lepu, gscpi, em_fx_vol) excluded here.
    'controls_macro_dense' : [
        c for c in [
            'vix', 'dvix',
            'gs10', 'tb3ms', 'term_spread',
            'tedrate', 'us_spread',
            'lreer',                   # 87.3% — starts 1994; accepted in dense spec
            'dcny_usd', 'cny_vol',
            'lgold', 'brent_wti_spread', 'lbdi',
            'lip',
            'us_cds_5y', 'china_cds_5y',   # included only if files were provided
        ]
        if c in df_enriched.columns and c not in zero_coverage
    ],

    # Sparse macro controls: <80% coverage.
    # Do NOT include in primary DoubleML — use only for robustness checks
    # on the restricted post-coverage subsample.
    # lepu     : starts 1997-02 (78.0%) — loses 85 obs from start of sample
    # gscpi    : starts 1998-02 (74.9%) — loses 96 obs
    # em_fx_vol: starts 2006-04 (49.5%) — loses ~195 obs; halves the sample
    'controls_macro_sparse' : [
        c for c in ['lepu', 'gscpi', 'em_fx_vol']
        if c in df_enriched.columns and c not in zero_coverage
    ],

    # Backwards-compatible full macro list (dense + sparse)
    'controls_macro' : [],   # populated below

    # Excluded variables (zero coverage in sample window)
    'EXCLUDED_zero_coverage' : zero_coverage,

    # Sparse variable list for reference
    'controls_sparse' : sparse_vars,
}

# Full macro list = dense + sparse (for robustness specs)
VAR_ROLES['controls_macro'] = (
    VAR_ROLES['controls_macro_dense'] + VAR_ROLES['controls_macro_sparse']
)

# PRIMARY spec: use this for DoubleML, causal forests, SHAP.
# Effective sample: ~336 obs (1994-02 to 2022-02, limited by lreer start).
VAR_ROLES['controls_all_ml_dense'] = (
    VAR_ROLES['controls_baseline']
    + VAR_ROLES['controls_alliance']
    + VAR_ROLES['controls_macro_dense']
)

# ROBUSTNESS spec: includes sparse vars.
# Effective sample: ~191 obs (2006-04 to 2022-02, limited by em_fx_vol start).
# Only use this for post-2006 sensitivity checks.
VAR_ROLES['controls_all_ml'] = (
    VAR_ROLES['controls_baseline']
    + VAR_ROLES['controls_alliance']
    + VAR_ROLES['controls_macro']
)

print('Variable inventory:')
for role, cols in VAR_ROLES.items():
    print(f'  {role:28s} ({len(cols):2d}): {cols}')

print(f'\nPRIMARY spec  (controls_all_ml_dense): {len(VAR_ROLES["controls_all_ml_dense"])} controls')
print(f'ROBUSTNESS spec (controls_all_ml)     : {len(VAR_ROLES["controls_all_ml"])} controls')
print(f'\nEffective obs — primary spec   : ~{df_enriched[VAR_ROLES["controls_macro_dense"]].dropna().shape[0]} (after dropping NaN rows)')
print(f'Effective obs — robustness spec: ~{df_enriched[VAR_ROLES["controls_macro"]].dropna().shape[0]} (after dropping NaN rows)')


Variable inventory:
  outcome                      ( 1): ['lwti']
  treatment                    ( 1): ['lpri']
  instrument                   ( 2): ['d2pri', 'd2pri_jp']
  controls_baseline            ( 4): ['llwip', 'dllgop', 'l2lwip', 'dl2lgop']
  controls_alliance            (11): ['dlpri_jp', 'dlpri_aus', 'dlpri_cds', 'dlpri_fra', 'dlpri_ger', 'dlpri_india', 'dlpri_indo', 'dlpri_pak', 'dlpri_rus', 'dlpri_vn', 'dlpri_uk']
  controls_macro_dense         (14): ['vix', 'dvix', 'gs10', 'tb3ms', 'term_spread', 'tedrate', 'us_spread', 'lreer', 'dcny_usd', 'cny_vol', 'lgold', 'brent_wti_spread', 'lbdi', 'lip']
  controls_macro_sparse        ( 2): ['lepu', 'em_fx_vol']
  controls_macro               (16): ['vix', 'dvix', 'gs10', 'tb3ms', 'term_spread', 'tedrate', 'us_spread', 'lreer', 'dcny_usd', 'cny_vol', 'lgold', 'brent_wti_spread', 'lbdi', 'lip', 'lepu', 'em_fx_vol']
  EXCLUDED_zero_coverage       ( 1): ['em_oas']
  controls_sparse              ( 2): ['lepu', 'em_fx_vol']
  controls_al

In [17]:
# =============================================================================
# VERIFICATION CELL – Double‑lagging check & control integrity
# Run after B7 (export) to confirm no unintended shifts
# =============================================================================

print("\n=== VERIFICATION: Control construction & lagging ===\n")

# 1. Check the lag of brent_wti_spread
#    Expected: Lag 1 (i.e., at time t, spread = lbrent_{t-1} - lwti_{t-1})
#    If double-lagged: spread uses t-2 values

# Load the latest df_enriched (already in memory after B7)
df_test = df_enriched.copy()

# Compute a theoretical "correct" spread (lagged once)
# Use original lwti (not lagged) and lbrent from the core df (pre‑lagging)
# We need to reconstruct lbrent from raw monthly_series saved before B4? Simpler:
# Use the spread we built in B3 (before any lagging) and lag it once manually.

# First, re‑extract the spread as it was before the global shift(1) in B4.
# We stored monthly_series['brent_wti_spread'] before B4 – but that variable
# may have been overwritten. Instead, recompute from scratch using the core lwti
# and the Brent series that existed before lagging.

# We'll use the original monthly_series stored before B4? Not saved.
# Safer: Use the fact that after B4, all new controls (including spread) are shifted.
# We can compare the spread in df_enriched with a manually recomputed spread
# that we lag only once.

# Recompute lbrent from the raw FRED file (still available in monthly_series before deletion?)
# Actually monthly_series lost 'lbrent' after B3. So we reload from raw file.

import pandas as pd
import numpy as np

# Reload Brent from raw file (same as in B2)
brent_d = pd.read_csv(f'{RAW}/brent_fred.csv', index_col=0, parse_dates=True, na_values=['.', 'NA', ''])
brent_d = ensure_datetimeindex(brent_d)
brent_d['DCOILBRENTEU'] = pd.to_numeric(brent_d['DCOILBRENTEU'], errors='coerce')
brent_m = to_monthly_mean(brent_d['DCOILBRENTEU'])
lbrent_raw = np.log(brent_m.replace(0, np.nan)).rename('lbrent')
lbrent_raw.index = lbrent_raw.index.to_period('M').to_timestamp()

# Align core lwti
lwti_core = df_core['lwti'].copy()
lwti_core.index = lwti_core.index.to_period('M').to_timestamp()

# Construct spread without any lag
spread_raw = (lbrent_raw - lwti_core).rename('brent_wti_spread_raw')

# Now lag it exactly once (this is what we intended)
spread_lag1 = spread_raw.shift(1)

# Compare with the spread in df_enriched (which may have been lagged twice)
spread_in_df = df_enriched['brent_wti_spread']

# Align indices
common_idx = spread_in_df.index.intersection(spread_lag1.index)
diff = spread_in_df.loc[common_idx] - spread_lag1.loc[common_idx]
max_diff = diff.abs().max()

print(f"1. brent_wti_spread verification:")
print(f"   Max difference between df_enriched['brent_wti_spread'] and manually lagged (once): {max_diff:.8f}")
if max_diff < 1e-6:
    print("   ✅ Spread is lagged exactly once (no double lag).")
else:
    print("   ❌ Spread differs from single‑lag version. Possible double lag or other issue.")
    # Check if it matches a double‑lagged version
    spread_lag2 = spread_raw.shift(2)
    diff2 = spread_in_df.loc[common_idx] - spread_lag2.loc[common_idx]
    if diff2.abs().max() < 1e-6:
        print("   → Spread matches a double‑lagged version (lagged twice in B4).")

# 2. Check that all new macro controls are lagged by exactly 1 month
#    We can examine the first few rows of each control vs. its raw original (if we had it).
#    A simpler check: ensure that the control series have no missing values at the very first period
#    (because lagging shifts them forward). But that's not definitive.
#    Instead, we compare the first non‑NaN value of each control with the raw source's
#    value at the previous month.

print("\n2. Checking lag length for other macro controls (sampling):")
macro_sample = ['vix', 'lgold', 'tb3ms', 'gs10', 'lip', 'lbdi']
all_correct = True
for c in macro_sample:
    # Reload raw daily series from original file and compute monthly value without lag
    if c == 'vix':
        raw_d = pd.read_csv(f'{RAW}/vix.csv', skiprows=lambda i: i in [1,2], index_col=0, parse_dates=True)
        raw_d = ensure_datetimeindex(raw_d)
        raw_m = to_monthly_mean(raw_d.iloc[:,0])
    elif c == 'lgold':
        raw_d = pd.read_csv(f'{RAW}/gold_monthly.csv', index_col=0, parse_dates=True)
        raw_m = to_monthly_last(raw_d.iloc[:,0])
        raw_m = np.log(raw_m.replace(0, np.nan))
    elif c in ['tb3ms', 'gs10', 'lip']:
        # For FRED series, reload from raw CSV (original levels)
        if c == 'lip':
            # indpro is the raw series; we need the level before log and diff
            raw_d = pd.read_csv(f'{RAW}/indpro.csv', index_col=0, parse_dates=True, na_values=['.', 'NA', ''])
            raw_d = ensure_datetimeindex(raw_d)
            raw_m = to_monthly_last(raw_d.iloc[:,0])
            raw_m = np.log(raw_m.replace(0, np.nan))
            raw_name = 'indpro'
        else:
            fred_map = {'tb3ms':'TB3MS', 'gs10':'GS10'}
            raw_d = pd.read_csv(f'{RAW}/{c}.csv', index_col=0, parse_dates=True, na_values=['.', 'NA', ''])
            raw_d = ensure_datetimeindex(raw_d)
            raw_m = to_monthly_last(raw_d.iloc[:,0])
            raw_name = c
    elif c == 'lbdi':
        raw_d = pd.read_csv(f'{RAW}/bdi_clean.csv', index_col=0, parse_dates=True)
        raw_m = to_monthly_mean(raw_d['bdi'])
        raw_m = np.log(raw_m.replace(0, np.nan))
    else:
        continue

    # Raw monthly series (no lag)
    raw_m.index = raw_m.index.to_period('M').to_timestamp()
    # Lag once manually
    manual_lag1 = raw_m.shift(1)
    # Get the series from df_enriched
    enriched_series = df_enriched[c].dropna()
    # Align indices
    common = enriched_series.index.intersection(manual_lag1.index)
    if len(common) == 0:
        print(f"   {c}: no overlapping dates – check")
        continue
    diff_c = (enriched_series.loc[common] - manual_lag1.loc[common]).abs().max()
    if diff_c > 1e-6:
        print(f"   {c}: max diff = {diff_c:.8f} → ❌ not exactly lagged once (or raw series not identical)")
        all_correct = False
    else:
        print(f"   {c}: ✅ lagged exactly once")

if all_correct:
    print("   All tested controls are lagged exactly once.")
else:
    print("   Some controls show mismatch. Check the raw data and lagging step in B4.")

# 3. Confirm that forbidden raw columns are absent
print("\n3. Forbidden columns check:")
forbidden = ['lbrent', 'cny_usd', 'em_fx_idx', 'reer', 'indpro', 'epu', 'bdi']
present = [col for col in forbidden if col in df_enriched.columns]
if not present:
    print("   ✅ No forbidden raw/level columns in df_enriched.")
else:
    print(f"   ❌ Still present: {present}")

# 4. Quick correlation check: brent_wti_spread vs lwti lagged
#    Should be low correlation after lagging (because spread is orthogonal by construction)
corr_spread = df_enriched['brent_wti_spread'].corr(df_enriched['lwti'].shift(1))
print(f"\n4. Correlation between brent_wti_spread and lwti_lagged: {corr_spread:.4f}")
print("   (If spread is properly lagged, correlation should be small in absolute value.)")

# 5. Effective sample size after dropping NaNs in dense spec
dense_vars = VAR_ROLES['controls_all_ml_dense']
n_eff = df_enriched[dense_vars].dropna().shape[0]
print(f"\n5. Effective observations for dense spec: {n_eff} (out of {len(df_enriched)})")
print(f"   Expected ~337 (due to lreer start 1994‑02). Got {n_eff}.")

print("\n=== Verification complete ===")


=== VERIFICATION: Control construction & lagging ===

1. brent_wti_spread verification:
   Max difference between df_enriched['brent_wti_spread'] and manually lagged (once): 0.00000000
   ✅ Spread is lagged exactly once (no double lag).

2. Checking lag length for other macro controls (sampling):
   vix: ✅ lagged exactly once
   lgold: ✅ lagged exactly once
   tb3ms: ✅ lagged exactly once
   gs10: ✅ lagged exactly once
   lip: ✅ lagged exactly once
   lbdi: ✅ lagged exactly once
   All tested controls are lagged exactly once.

3. Forbidden columns check:
   ✅ No forbidden raw/level columns in df_enriched.

4. Correlation between brent_wti_spread and lwti_lagged: 0.6395
   (If spread is properly lagged, correlation should be small in absolute value.)

5. Effective observations for dense spec: 337 (out of 386)
   Expected ~337 (due to lreer start 1994‑02). Got 337.

=== Verification complete ===


In [18]:
macro_vars = [v for v in VAR_ROLES['controls_macro'] if v in df_enriched.columns]

### B7. Export

In [19]:
OUT_CSV   = f'{PROC}/feature_matrix.csv'
OUT_ROLES = f'{PROC}/var_roles.json'

# ── SAFETY ASSERTIONS before every export ────────────────────────────────────
# Raw levels that must not appear (either collinear with outcome or non-stationary)
forbidden_cols = {
    'lbrent'   : 'collinear with lwti (ρ≈0.99)',
    'cny_usd'  : 'I(1) level — use dcny_usd and cny_vol instead',
    'em_fx_idx': 'raw index level — use em_fx_vol instead',
    'reer'     : 'I(1) level — use lreer instead',
    'indpro'   : 'raw level — use lip instead',
    'epu'      : 'raw level — use lepu instead',
    'bdi'      : 'raw level — use lbdi instead',
}
for col, reason in forbidden_cols.items():
    assert col not in df_enriched.columns, \
        f"CRITICAL: '{col}' in export ({reason}). Re-run from B3."
print('✓ No forbidden raw/level columns in df_enriched')

# All dense-spec controls must exist
missing_dense = [c for c in VAR_ROLES['controls_all_ml_dense']
                 if c not in df_enriched.columns]
assert not missing_dense, f"controls_all_ml_dense references missing columns: {missing_dense}"
print('✓ All controls_all_ml_dense columns present')

# All robustness controls must exist
missing_all = [c for c in VAR_ROLES['controls_all_ml']
               if c not in df_enriched.columns]
assert not missing_all, f"controls_all_ml references missing columns: {missing_all}"
print('✓ All controls_all_ml columns present')

# No Stata intermediates in either control list
stata_intermediates = ['lgop', 'lwip', 'pri', 'pri_jp', 'u_gop', 'u_wip',
                       'lpri_aus','lpri_cds','lpri_fra','lpri_ger','lpri_india',
                       'lpri_indo','lpri_pak','lpri_rus','lpri_uk','lpri_vn']
for spec_name, spec_cols in [('dense', VAR_ROLES['controls_all_ml_dense']),
                              ('all_ml', VAR_ROLES['controls_all_ml'])]:
    leaked = [c for c in spec_cols if c in stata_intermediates]
    assert not leaked, f"Stata intermediates in {spec_name}: {leaked}"
print('✓ No Stata intermediates in either control list')

# Export
df_enriched.to_csv(OUT_CSV)
with open(OUT_ROLES, 'w') as fh:
    json.dump({k: list(v) for k, v in VAR_ROLES.items()}, fh, indent=2)

print(f'\nSaved: {OUT_CSV}')
print(f'Saved: {OUT_ROLES}')
print(f'Feature matrix shape: {df_enriched.shape}')

# Summary of what's in the CSV
ml_cols    = set(VAR_ROLES['controls_all_ml'])
dense_cols = set(VAR_ROLES['controls_all_ml_dense'])
csv_cols   = list(df_enriched.columns)
extra      = [c for c in csv_cols
              if c not in ml_cols
              and c not in ['lwti','lpri','d2pri','d2pri_jp','Period']]
print(f'\nColumns in controls_all_ml_dense : {len(dense_cols)}')
print(f'Columns in controls_all_ml       : {len(ml_cols)}')
print(f'Extra columns in CSV (not in either ML spec): {len(extra)}')
print(f'  → These are Stata pass-throughs (lpri_*, pri_*, llgop, etc.)')
print(f'  → Use var_roles.json to select the correct feature set — do not use all CSV columns.')


✓ No forbidden raw/level columns in df_enriched
✓ All controls_all_ml_dense columns present
✓ All controls_all_ml columns present
✓ No Stata intermediates in either control list

Saved: C:\Users\HP\Desktop\replication+contribution\data\02_features/feature_matrix.csv
Saved: C:\Users\HP\Desktop\replication+contribution\data\02_features/var_roles.json
Feature matrix shape: (386, 67)

Columns in controls_all_ml_dense : 29
Columns in controls_all_ml       : 31
Extra columns in CSV (not in either ML spec): 31
  → These are Stata pass-throughs (lpri_*, pri_*, llgop, etc.)
  → Use var_roles.json to select the correct feature set — do not use all CSV columns.


---
## Appendix — Descriptive Statistics & Correlations

In [20]:
macro_vars = [v for v in VAR_ROLES['controls_macro'] if v in df_enriched.columns]
if macro_vars:
    print('=== Descriptive statistics (macro controls) ===')
    display(df_enriched[macro_vars].describe().round(3))

=== Descriptive statistics (macro controls) ===


,vix,dvix,gs10,tb3ms,term_spread,tedrate,us_spread,lreer,dcny_usd,cny_vol,lgold,brent_wti_spread,lbdi,lip,lepu,em_fx_vol
count,385.000,384.000,385.000,385.000,385.000,385.000,385.000,337.000,385.000,385.000,385.000,385.000,385.000,385.000,301.000,191.000
mean,19.499,-0.002,4.294,2.541,1.753,0.466,2.349,4.526,0.001,0.006,6.473,0.655,9.581,4.477,4.749,0.013
std,7.657,4.154,2.038,2.233,1.088,0.356,0.729,0.082,0.022,0.014,0.674,0.279,0.575,0.161,0.457,0.009
min,10.125,-16.283,0.620,0.010,-0.530,0.060,1.300,4.362,-0.035,0.000,5.545,0.137,7.938,4.100,3.948,0.001
25%,13.975,-1.730,2.490,0.160,0.880,0.230,1.790,4.454,-0.003,0.000,5.876,0.384,9.250,4.435,4.365,0.007
50%,17.637,-0.265,4.200,2.070,1.690,0.380,2.210,4.540,-0.000,0.002,6.310,0.658,9.663,4.530,4.733,0.011
75%,23.141,1.187,5.900,4.810,2.660,0.570,2.760,4.586,0.000,0.006,7.148,0.920,10.048,4.597,5.059,0.018
max,62.639,38.108,8.890,7.900,3.760,3.150,6.100,4.689,0.405,0.116,7.585,1.099,10.369,4.645,6.036,0.044


In [21]:
# Correlation with log WTI — exploratory only, not causal
if macro_vars:
    print('=== Correlations with log WTI (exploratory) ===')
    corr = (
        df_enriched[macro_vars + ['lwti']]
        .corr()['lwti']
        .drop('lwti')
        .sort_values(key=abs, ascending=False)
        .round(3)
    )
    print(corr.to_string())

=== Correlations with log WTI (exploratory) ===
lgold               0.653
brent_wti_spread    0.640
lip                 0.558
lbdi                0.551
lreer              -0.551
tb3ms              -0.500
gs10               -0.499
us_spread           0.248
em_fx_vol          -0.235
dcny_usd           -0.125
vix                -0.120
term_spread         0.091
lepu                0.085
tedrate            -0.077
cny_vol            -0.067
dvix               -0.032
